# 01. Скачивание данных и микширование (v7)

**Цель:** скачать целевые звуки и фоны леса из открытых датасетов, нормализовать и создать миксы с разным SNR.

**Изменения v5 -> v7:**
- **source_target_file** — каждый микс хранит имя исходного WAV (для data leakage fix в NB02)
- **background-only миксы** — 300 чистых фонов для FPR-теста (TODO-02)
- Итого: 2100 target-миксов + 300 bg-only = **2400 записей** в manifest

**Датасеты:**
- ESC-50 (HuggingFace) — chainsaw, engine, crackling_fire + фоны (rain, wind, birds, crickets, pouring_water)
- UrbanSound8K (HuggingFace) — gun_shot, engine_idling
- FSC22 (Kaggle) — axe

**Результат:** `data/mixed/` с 2400 файлами + `manifest.csv`

In [64]:
%pip install -q -r ../requirements.txt

In [65]:
import os
import numpy as np
import librosa
import soundfile as sf
import pandas as pd
from pathlib import Path

# Корневая директория проекта
ROOT = Path("../data")
TARGETS_DIR = ROOT / "targets"
BG_DIR = ROOT / "backgrounds"
MIXED_DIR = ROOT / "mixed"

TARGET_SR = 16000  # YAMNet требует 16kHz

for d in [TARGETS_DIR, BG_DIR, MIXED_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Targets: {TARGETS_DIR}")
print(f"Backgrounds: {BG_DIR}")
print(f"Mixed: {MIXED_DIR}")

Targets: ../data/targets
Backgrounds: ../data/backgrounds
Mixed: ../data/mixed


## Вспомогательные функции

In [66]:
def extract_audio_from_hf(row):
    """Извлечь аудио из HuggingFace dataset row -> (np.array, sr)."""
    audio_data = row["audio"]
    return np.array(audio_data["array"], dtype=np.float32), audio_data["sampling_rate"]


def normalize_audio(audio: np.ndarray, sr: int) -> np.ndarray:
    """Ресемплинг на TARGET_SR + peak-нормализация."""
    if sr != TARGET_SR:
        audio = librosa.resample(audio, orig_sr=sr, target_sr=TARGET_SR)
    peak = np.max(np.abs(audio))
    if peak > 0:
        audio = audio / peak
    return audio


def save_wav(audio: np.ndarray, path: Path):
    """Сохранить WAV 16kHz mono."""
    path.parent.mkdir(parents=True, exist_ok=True)
    sf.write(str(path), audio, TARGET_SR)


def mix_with_snr(signal: np.ndarray, noise: np.ndarray, snr_db: float) -> np.ndarray:
    """Микширование сигнала с шумом при заданном SNR (дБ).

    v5 Fix 4.4: нормализация ДО микширования вместо post-clipping.
    v4: `if peak > 1.0: mixed /= peak` искажал реальный SNR после микширования.
    v5: нормализуем signal и noise до unit RMS перед расчётом, затем масштабируем.
    """
    # Выровнять длину
    if len(noise) < len(signal):
        noise = np.tile(noise, int(np.ceil(len(signal) / len(noise))))
    noise = noise[:len(signal)]

    rms_signal = np.sqrt(np.mean(signal ** 2))
    rms_noise = np.sqrt(np.mean(noise ** 2))

    if rms_noise == 0 or rms_signal == 0:
        return signal.copy()

    # v5 Fix 4.4: нормализуем signal к unit peak ДО микширования
    sig_peak = np.max(np.abs(signal))
    if sig_peak > 0:
        signal_norm = signal / sig_peak * 0.7  # headroom для микширования
    else:
        signal_norm = signal.copy()

    rms_signal_norm = np.sqrt(np.mean(signal_norm ** 2))
    target_rms_noise = rms_signal_norm / (10 ** (snr_db / 20))
    noise_scaled = noise * (target_rms_noise / rms_noise)

    mixed = signal_norm + noise_scaled

    # Soft clip вместо hard normalization
    peak = np.max(np.abs(mixed))
    if peak > 1.0:
        mixed = np.tanh(mixed)  # v5: soft clipping preserves relative levels
    return mixed


# Параметры микширования
SNR_LEVELS = [20, 15, 10, 5, 0, -5, -10]
SAMPLES_PER_COMBO = 10  # v3: увеличено с 3 до 10
rng = np.random.RandomState(42)

print("Вспомогательные функции определены.")
print(f"SNR levels: {SNR_LEVELS}")
print(f"Samples per combo: {SAMPLES_PER_COMBO}")
print(f"Ожидаемое число миксов: 5 targets × 6 backgrounds × 7 SNR × {SAMPLES_PER_COMBO} = {5*6*7*SAMPLES_PER_COMBO}")

Вспомогательные функции определены.
SNR levels: [20, 15, 10, 5, 0, -5, -10]
Samples per combo: 10
Ожидаемое число миксов: 5 targets × 6 backgrounds × 7 SNR × 10 = 2100


## RIR-аугментация "динамик → микрофон"

Симуляция передачи аудио через динамик + микрофон с реверберацией, lowpass и AGC.

In [ ]:
import scipy.signal

def generate_synthetic_rir(sr=16000, rt60=0.3):
    """Синтетический Room Impulse Response (экспоненциальный decay)."""
    n_samples = int(sr * rt60)
    t = np.arange(n_samples) / sr
    decay = np.exp(-6.9 * t / rt60)  # -60 dB при t=RT60
    noise = rng.randn(n_samples)
    rir = noise * decay
    rir[0] = 1.0  # direct path
    return rir / np.max(np.abs(rir))


def apply_speaker_mic_aug(audio, sr=16000):
    """Симуляция передачи динамик → микрофон.

    1. RIR-конволюция (реверберация 0.1–0.5 с)
    2. Lowpass 14 kHz (предел динамиков)
    3. AGC-компрессия (сглаживание динамического диапазона)
    """
    # 1. RIR
    rt60 = rng.uniform(0.1, 0.5)
    rir = generate_synthetic_rir(sr, rt60)
    convolved = scipy.signal.fftconvolve(audio, rir, mode='full')[:len(audio)]

    # 2. Lowpass 14 kHz
    nyq = sr / 2
    cutoff = min(14000, nyq - 100)
    sos = scipy.signal.butter(4, cutoff / nyq, btype='low', output='sos')
    filtered = scipy.signal.sosfilt(sos, convolved)

    # 3. AGC
    rms = np.sqrt(np.mean(filtered**2))
    if rms > 0:
        filtered = filtered / (rms + 0.01) * 0.3

    return filtered.astype(np.float32)


print("RIR augmentation functions defined.")

## Секция 1 — ESC-50 (HuggingFace)

Извлекаем:
- **Targets:** chainsaw (40), engine (40), crackling_fire (40)
- **Backgrounds:** rain (40), wind (40), chirping_birds (40), crickets→forest_night (40), pouring_water→river (40)

In [67]:
from datasets import load_dataset

print("Загружаем ESC-50...")
esc50 = load_dataset("ashraq/esc50", split="train")
print(f"Загружено {len(esc50)} сэмплов")

# Маппинг категорий ESC-50 -> наши папки
ESC50_TARGETS = {
    "chainsaw": "chainsaw",
    "engine": "engine",
    "crackling_fire": "fire",
}
ESC50_BACKGROUNDS = {
    "rain": "rain",
    "wind": "wind",
    "chirping_birds": "birds",
    "crickets": "forest_night",
    "pouring_water": "river",
}

# Все нужные категории
all_categories = {**ESC50_TARGETS, **ESC50_BACKGROUNDS}

counts = {k: 0 for k in all_categories}

for row in esc50:
    category = row["category"]
    if category not in all_categories:
        continue
    
    folder_name = all_categories[category]
    is_target = category in ESC50_TARGETS
    base_dir = TARGETS_DIR if is_target else BG_DIR
    out_dir = base_dir / folder_name
    
    audio_arr, sr = extract_audio_from_hf(row)
    audio_norm = normalize_audio(audio_arr, sr)
    
    idx = counts[category]
    save_wav(audio_norm, out_dir / f"{folder_name}_esc50_{idx:03d}.wav")
    counts[category] += 1

print("\nESC-50 извлечение завершено:")
for cat, cnt in counts.items():
    print(f"  {cat} -> {all_categories[cat]}: {cnt} файлов")

Загружаем ESC-50...


Repo card metadata block was not found. Setting CardData to empty.


Загружено 2000 сэмплов

ESC-50 извлечение завершено:
  chainsaw -> chainsaw: 40 файлов
  engine -> engine: 40 файлов
  crackling_fire -> fire: 40 файлов
  rain -> rain: 40 файлов
  wind -> wind: 40 файлов
  chirping_birds -> birds: 40 файлов
  crickets -> forest_night: 40 файлов
  pouring_water -> river: 40 файлов


## Секция 2 — UrbanSound8K (HuggingFace)

Извлекаем:
- **gun_shot** (classID=6, до 50) → `targets/gunshot/`
- **engine_idling** (classID=5, до 50 для дополнения) → `targets/engine/`

In [68]:
print("Загружаем UrbanSound8K...")
us8k = load_dataset("danavery/urbansound8K", split="train")
print(f"Загружено {len(us8k)} сэмплов")

# classID: 5=engine_idling, 6=gun_shot
US8K_TARGETS = {
    6: {"folder": "gunshot", "max": 50, "count": 0},
    5: {"folder": "engine", "max": 50, "count": 0},
}

for row in us8k:
    class_id = row["classID"]
    if class_id not in US8K_TARGETS:
        continue
    
    info = US8K_TARGETS[class_id]
    if info["count"] >= info["max"]:
        continue
    
    audio_arr, sr = extract_audio_from_hf(row)
    audio_norm = normalize_audio(audio_arr, sr)
    
    out_dir = TARGETS_DIR / info["folder"]
    save_wav(audio_norm, out_dir / f"{info['folder']}_us8k_{info['count']:03d}.wav")
    info["count"] += 1

print("\nUrbanSound8K извлечение завершено:")
for class_id, info in US8K_TARGETS.items():
    print(f"  classID={class_id} -> {info['folder']}: {info['count']} файлов")

Загружаем UrbanSound8K...
Загружено 8732 сэмплов

UrbanSound8K извлечение завершено:
  classID=6 -> gunshot: 50 файлов
  classID=5 -> engine: 50 файлов


## Секция 3 — FSC22 (Kaggle)

Извлекаем:
- **axe** (до 75 шт.) → `targets/axe/`

Требуется Kaggle API token (`~/.kaggle/kaggle.json`). Если нет доступа — скачайте вручную с https://www.kaggle.com/datasets/irmiot22/fsc22-dataset

In [69]:
import kagglehub
import glob

AXE_DIR = TARGETS_DIR / "axe"
AXE_DIR.mkdir(parents=True, exist_ok=True)

try:
    print("Скачиваем FSC22 с Kaggle...")
    fsc22_path = kagglehub.dataset_download("irmiot22/fsc22-dataset")
    print(f"FSC22 скачан в: {fsc22_path}")

    # FSC22 хранит файлы как {CLASS_ID}_{SEQ_ID}.wav в плоской папке
    # Class 10 = Axe, Class 16 = WoodChop
    audio_dir_pattern = os.path.join(fsc22_path, "**", "*.wav")
    all_wavs = glob.glob(audio_dir_pattern, recursive=True)

    # Фильтруем по префиксу класса (10_ = Axe, 16_ = WoodChop)
    axe_files = [f for f in all_wavs if os.path.basename(f).startswith(("10_", "16_"))]
    axe_files = axe_files[:75]  # до 75 штук

    print(f"Найдено {len(axe_files)} axe/woodchop файлов")

    for i, fpath in enumerate(axe_files):
        audio, sr = librosa.load(fpath, sr=None)
        audio_norm = normalize_audio(audio, sr)
        save_wav(audio_norm, AXE_DIR / f"axe_fsc22_{i:03d}.wav")

    print(f"Сохранено {len(axe_files)} axe-файлов")

except Exception as e:
    print(f"Ошибка загрузки FSC22: {e}")
    print("\nFallback: скачайте датасет вручную с Kaggle и распакуйте в audio_poc/data/")
    print("URL: https://www.kaggle.com/datasets/irmiot22/fsc22-dataset")
    print("Затем перезапустите эту ячейку.")

Скачиваем FSC22 с Kaggle...
FSC22 скачан в: /Users/user/.cache/kagglehub/datasets/irmiot22/fsc22-dataset/versions/1
Найдено 75 axe/woodchop файлов
Сохранено 75 axe-файлов


## Секция 3b — Silence из FSC22

FSC22 Class 6 = Silence (75 файлов, префикс `6_`)

In [70]:
# Извлекаем Silence из FSC22 (Class 6, префикс "6_")
SILENCE_DIR = BG_DIR / "silence"
SILENCE_DIR.mkdir(parents=True, exist_ok=True)

try:
    fsc22_path = kagglehub.dataset_download("irmiot22/fsc22-dataset")
    all_wavs = glob.glob(os.path.join(fsc22_path, "**", "*.wav"), recursive=True)
    silence_files = [f for f in all_wavs if os.path.basename(f).startswith("6_")]

    print(f"Найдено {len(silence_files)} silence файлов")
    for i, fpath in enumerate(silence_files):
        audio, sr = librosa.load(fpath, sr=None)
        audio_norm = normalize_audio(audio, sr)
        save_wav(audio_norm, SILENCE_DIR / f"silence_fsc22_{i:03d}.wav")
    print(f"Сохранено {len(silence_files)} silence файлов")

except Exception as e:
    print(f"Ошибка: {e}")

Найдено 75 silence файлов
Сохранено 75 silence файлов


## Секция 3c — Птицы по видам (Bird songs from Europe, Kaggle)

Добавляем разнообразие видов в фон `birds/`. Источник: `monogenea/birdsongs-from-europe`.
Файлы MP3, именование: `Genus-species-RecordingID.mp3`

Виды (по доступности): Tawny Owl, Boreal Owl, Eagle Owl, Chaffinch, Robin, Cuckoo — по 9-14 файлов.

## Секция 3d — Разведка FSC22: все классы

Исследуем ВСЕ классы FSC22 для поиска chainsaw-подобных звуков.

In [ ]:
# === Разведка FSC22: все классы ===
import kagglehub, glob, os, re
from collections import Counter

fsc22_path = kagglehub.dataset_download("irmiot22/fsc22-dataset")
all_wavs = glob.glob(os.path.join(fsc22_path, "**", "*.wav"), recursive=True)

# Извлечь prefix (class_id) из имён файлов
prefixes = [os.path.basename(f).split("_")[0] for f in all_wavs]
counts = Counter(prefixes)

print(f"Всего файлов: {len(all_wavs)}")
print(f"\nКлассы FSC22 (prefix → count):")
for prefix, count in sorted(counts.items(), key=lambda x: int(x[0]) if x[0].isdigit() else 999):
    # Покажем примеры файлов для каждого класса
    examples = [os.path.basename(f) for f in all_wavs if os.path.basename(f).startswith(f"{prefix}_")][:3]
    print(f"  Class {prefix}: {count} файлов  |  Примеры: {examples}")

## Секция 3e — FSC22 Chainsaw (если найден)

Загружаем chainsaw-подобные файлы из FSC22 по найденному CLASS_ID.

In [ ]:
# === FSC22 Chainsaw (подставить правильный CLASS_ID после разведки) ===
CHAINSAW_CLASS_ID = "???"  # <-- подставить после разведки
CHAINSAW_DIR = TARGETS_DIR / "chainsaw"

chainsaw_files = [f for f in all_wavs if os.path.basename(f).startswith(f"{CHAINSAW_CLASS_ID}_")]
print(f"Найдено {len(chainsaw_files)} chainsaw файлов в FSC22")

existing = len(list(CHAINSAW_DIR.glob("*.wav")))
for i, fpath in enumerate(chainsaw_files):
    audio, sr = librosa.load(fpath, sr=None)
    audio_norm = normalize_audio(audio, sr)
    save_wav(audio_norm, CHAINSAW_DIR / f"chainsaw_fsc22_{i:03d}.wav")

print(f"Chainsaw: было {existing}, стало {existing + len(chainsaw_files)}")

In [71]:
# Птицы по видам из Bird songs from Europe
BIRDS_DIR = BG_DIR / "birds"  # добавляем к существующим ESC-50 birds

try:
    print("Скачиваем Bird songs from Europe...")
    birds_path = kagglehub.dataset_download("monogenea/birdsongs-from-europe")
    print(f"Скачан в: {birds_path}")

    mp3_dir = os.path.join(birds_path, "mp3")
    if not os.path.exists(mp3_dir):
        # Файлы могут быть в корне
        mp3_dir = birds_path

    # Целевые виды (Genus-species префикс -> русское название)
    BIRD_SPECIES = {
        "Strix-aluco": "tawny_owl",
        "Aegolius-funereus": "boreal_owl",
        "Bubo-bubo": "eagle_owl",
        "Fringilla-coelebs": "chaffinch",
        "Erithacus-rubecula": "robin",
        "Cuculus-canorus": "cuckoo",
    }

    all_mp3s = glob.glob(os.path.join(mp3_dir, "*.mp3"))
    bird_count = 0

    for fpath in sorted(all_mp3s):
        basename = os.path.basename(fpath)
        for prefix, species_name in BIRD_SPECIES.items():
            if basename.startswith(prefix):
                try:
                    audio, sr = librosa.load(fpath, sr=None, duration=10.0)  # max 10 сек
                    audio_norm = normalize_audio(audio, sr)
                    save_wav(audio_norm, BIRDS_DIR / f"bird_{species_name}_{bird_count:03d}.wav")
                    bird_count += 1
                except Exception as e:
                    print(f"  Ошибка {basename}: {e}")
                break

    print(f"\nСохранено {bird_count} bird файлов по видам в backgrounds/birds/")

except Exception as e:
    print(f"Ошибка загрузки Bird songs: {e}")
    print("Продолжаем с существующими ESC-50 birds.")

Скачиваем Bird songs from Europe...
Скачан в: /Users/user/.cache/kagglehub/datasets/monogenea/birdsongs-from-europe/versions/3

Сохранено 258 bird файлов по видам в backgrounds/birds/


## Сводка по скачанным данным

In [72]:
print("Сводка по скачанным данным:")
print("=" * 50)

total_targets = 0
total_backgrounds = 0

print("\nTargets:")
for name in ["chainsaw", "gunshot", "engine", "axe", "fire"]:
    d = TARGETS_DIR / name
    if d.exists():
        n = len(list(d.glob("*.wav")))
        total_targets += n
        print(f"  {name}: {n} файлов")
    else:
        print(f"  {name}: НЕ НАЙДЕНА")

print("\nBackgrounds:")
for name in ["rain", "wind", "birds", "forest_night", "river", "silence"]:
    d = BG_DIR / name
    if d.exists():
        n = len(list(d.glob("*.wav")))
        total_backgrounds += n
        print(f"  {name}: {n} файлов")
    else:
        print(f"  {name}: НЕ НАЙДЕНА")

print(f"\nИтого: {total_targets} target + {total_backgrounds} background = {total_targets + total_backgrounds} файлов")

Сводка по скачанным данным:

Targets:
  chainsaw: 40 файлов
  gunshot: 50 файлов
  engine: 90 файлов
  axe: 75 файлов
  fire: 40 файлов

Backgrounds:
  rain: 40 файлов
  wind: 40 файлов
  birds: 298 файлов
  forest_night: 40 файлов
  river: 40 файлов
  silence: 75 файлов

Итого: 295 target + 533 background = 828 файлов


## Секция 4 — Микширование с SNR (v3: 2100 миксов)

Для каждой комбинации (target × background × SNR) создаём **10** миксов.

- **5 targets** × **6 backgrounds** × **7 SNR** × **10 сэмплов** = **2100 миксов**
- Backgrounds: rain, wind, birds (с видами), forest_night, river, silence
- SNR: `[20, 15, 10, 5, 0, -5, -10]` дБ
- Шум масштабируется, сигнал остаётся (корректно для SNR-тестов)

In [73]:
# Загрузить все файлы в память
def load_all_wavs(directory: Path) -> tuple[list[np.ndarray], list[str]]:
    """Загрузить все WAV-файлы из директории. Возвращает (audio_list, filename_list)."""
    wavs = []
    names = []
    for f in sorted(directory.glob("*.wav")):
        audio, _ = librosa.load(str(f), sr=TARGET_SR)
        wavs.append(audio)
        names.append(f.name)
    return wavs, names

# Загружаем targets
target_names = ["chainsaw", "gunshot", "engine", "axe", "fire"]
targets = {}
target_filenames = {}  # v6: idx -> filename для data leakage fix

for name in target_names:
    d = TARGETS_DIR / name
    if d.exists():
        wavs, names = load_all_wavs(d)
        if wavs:
            targets[name] = wavs
            target_filenames[name] = names
            print(f"Target '{name}': {len(wavs)} файлов загружено")
        else:
            print(f"WARNING: {name} — 0 файлов, пропускаем")
    else:
        print(f"WARNING: {d} не найдена, пропускаем {name}")

# Загружаем backgrounds (включая silence)
bg_names = ["rain", "wind", "birds", "forest_night", "river", "silence"]
backgrounds = {}
for name in bg_names:
    d = BG_DIR / name
    if d.exists():
        wavs, _ = load_all_wavs(d)
        if wavs:
            backgrounds[name] = wavs
            print(f"Background '{name}': {len(wavs)} файлов загружено")
        else:
            print(f"WARNING: {name} — 0 файлов, пропускаем")
    else:
        print(f"WARNING: {d} не найдена, пропускаем {name}")

Target 'chainsaw': 40 файлов загружено
Target 'gunshot': 50 файлов загружено
Target 'engine': 90 файлов загружено
Target 'axe': 75 файлов загружено
Target 'fire': 40 файлов загружено
Background 'rain': 40 файлов загружено
Background 'wind': 40 файлов загружено
Background 'birds': 298 файлов загружено
Background 'forest_night': 40 файлов загружено
Background 'river': 40 файлов загружено
Background 'silence': 75 файлов загружено


In [74]:
# Генерация миксов (v6: + source_target_file)
manifest_rows = []
mix_count = 0

for target_name, target_wavs in targets.items():
    for bg_name, bg_wavs in backgrounds.items():
        for snr_db in SNR_LEVELS:
            for s in range(SAMPLES_PER_COMBO):
                # Случайный выбор сэмплов
                t_idx = rng.randint(0, len(target_wavs))
                b_idx = rng.randint(0, len(bg_wavs))
                
                signal = target_wavs[t_idx]
                noise = bg_wavs[b_idx]
                
                # v8: 30% миксов получают speaker-mic аугментацию
                if rng.random() < 0.3:
                    signal = apply_speaker_mic_aug(signal, TARGET_SR)
                
                mixed = mix_with_snr(signal, noise, snr_db)
                
                filename = f"{target_name}_{bg_name}_snr{snr_db:+d}_s{s:02d}.wav"
                save_wav(mixed, MIXED_DIR / filename)
                
                manifest_rows.append({
                    "filename": filename,
                    "target_class": target_name,
                    "background_class": bg_name,
                    "snr_db": snr_db,
                    "target_idx": t_idx,
                    "background_idx": b_idx,
                    "source_target_file": target_filenames[target_name][t_idx],  # v6
                })
                mix_count += 1

print(f"\nСоздано {mix_count} target-миксов")


Создано 2100 target-миксов


In [75]:
# v6 TODO-02: Генерация 300 background-only миксов для FPR-теста
N_BG_ONLY = 300
bg_only_count = 0

for i in range(N_BG_ONLY):
    # Случайный выбор фона
    bg_name = bg_names[rng.randint(0, len(bg_names))]
    bg_wavs = backgrounds[bg_name]
    b_idx = rng.randint(0, len(bg_wavs))
    bg_audio = bg_wavs[b_idx]

    # 5-секундный отрезок
    target_len = 5 * TARGET_SR  # 80000 samples
    if len(bg_audio) > target_len:
        start = rng.randint(0, len(bg_audio) - target_len)
        bg_clip = bg_audio[start:start + target_len]
    else:
        bg_clip = bg_audio.copy()

    # 50% шанс микса двух фонов для разнообразия
    if rng.random() < 0.5:
        bg_name2 = bg_names[rng.randint(0, len(bg_names))]
        bg_wavs2 = backgrounds[bg_name2]
        bg2 = bg_wavs2[rng.randint(0, len(bg_wavs2))]
        snr_mix = rng.uniform(-3, 3)
        bg_clip = mix_with_snr(bg_clip, bg2, snr_mix)

    filename = f"bgonly_{bg_name}_s{i:03d}.wav"
    save_wav(bg_clip, MIXED_DIR / filename)

    manifest_rows.append({
        "filename": filename,
        "target_class": "background",
        "background_class": bg_name,
        "snr_db": 0,
        "target_idx": -1,
        "background_idx": b_idx,
        "source_target_file": "none",
    })
    bg_only_count += 1

print(f"Создано {bg_only_count} background-only миксов")
print(f"Итого в manifest: {len(manifest_rows)} записей (2100 target + {bg_only_count} bg-only)")

Создано 300 background-only миксов
Итого в manifest: 2400 записей (2100 target + 300 bg-only)


In [76]:
# Сохраняем manifest
manifest_df = pd.DataFrame(manifest_rows)
manifest_path = MIXED_DIR / "manifest.csv"
manifest_df.to_csv(manifest_path, index=False)

print(f"Manifest сохранён: {manifest_path}")
print(f"Всего миксов: {len(manifest_df)}")
print(f"\nРаспределение по targets:")
print(manifest_df["target_class"].value_counts().to_string())
print(f"\nРаспределение по backgrounds:")
print(manifest_df["background_class"].value_counts().to_string())
print(f"\nРаспределение по SNR:")
print(manifest_df["snr_db"].value_counts().sort_index().to_string())

Manifest сохранён: ../data/mixed/manifest.csv
Всего миксов: 2400

Распределение по targets:
target_class
chainsaw      420
gunshot       420
engine        420
axe           420
fire          420
background    300

Распределение по backgrounds:
background_class
silence         413
forest_night    410
birds           401
river           398
rain            395
wind            383

Распределение по SNR:
snr_db
-10    300
-5     300
 0     600
 5     300
 10    300
 15    300
 20    300


In [77]:
# Финальная проверка
mixed_files = list(MIXED_DIR.glob("*.wav"))
print(f"\nФайлов в data/mixed/: {len(mixed_files)} WAV + manifest.csv")

print("\nПримеры target-миксов:")
target_files = [f for f in sorted(mixed_files) if not f.name.startswith("bgonly_")]
for f in target_files[:5]:
    print(f"  {f.name}")
print("  ...")

bg_files = [f for f in sorted(mixed_files) if f.name.startswith("bgonly_")]
print(f"\nBackground-only примеры ({len(bg_files)} файлов):")
for f in bg_files[:3]:
    print(f"  {f.name}")
print("  ...")

print(f"\nНоутбук 01 завершён. Переходите к 02_yamnet_test.ipynb")


Файлов в data/mixed/: 2400 WAV + manifest.csv

Примеры target-миксов:
  axe_birds_snr+0_s00.wav
  axe_birds_snr+0_s01.wav
  axe_birds_snr+0_s02.wav
  axe_birds_snr+0_s03.wav
  axe_birds_snr+0_s04.wav
  ...

Background-only примеры (300 файлов):
  bgonly_birds_s002.wav
  bgonly_birds_s004.wav
  bgonly_birds_s007.wav
  ...

Ноутбук 01 завершён. Переходите к 02_yamnet_test.ipynb
